# Pearson Correlation Evaluation for Mean Shift Ablation

This notebook decodes embedding predictions to gene expression space and compares:
1. State model predictions
2. Mean shift baseline
3. Control passthrough baseline

Against ground truth gene expression using Pearson correlation.

## Configuration

In [ ]:
from pathlib import Path
import anndata as ad
import numpy as np
import torch
import yaml
import os
from tqdm import tqdm
import logging

logging.basicConfig(level=logging.INFO)
logger = logging.getLogger(__name__)

In [ ]:
# File paths
REAL_H5AD = "data/real.h5ad"
PRED_STATE = "data/pred.h5ad"
PRED_MEAN_SHIFT = "mean_shift_results/pred_lms.h5ad"
PRED_CONTROL = "mean_shift_results/pred_control_passthrough.h5ad"

# State model checkpoint
MODEL_DIR = "/tahoe/drive_3/ANALYSIS/analysis_190/Code/train_state_tx/experiments/tahoe_state_tx_20250821_045711_MFM3B_hvg_full/"
CHECKPOINT = "final.ckpt"

# Output directory for decoded predictions
OUTPUT_DIR = Path("mean_shift_results")
OUTPUT_DIR.mkdir(exist_ok=True)

# Evaluation parameters
CONTROL_PERT = "PBS"
PERT_COL = "cytokine"
CELLTYPE_COL = "donor"

# Embedding keys
EMBED_KEY_STATE = "tahoe_x1_3b"  # State model uses this embedding
EMBED_KEY_BASELINES = "model_preds"  # Mean shift and control use this

# Decoding parameters
BATCH_SIZE = 512
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

print(f"Using device: {DEVICE}")

## Step 1: Load State Model and Decoder

In [ ]:
def load_state_model(model_dir, checkpoint_name):
    """Load State TX model with decoder."""
    # Load config
    config_path = os.path.join(model_dir, "config.yaml")
    with open(config_path, 'r') as f:
        cfg = yaml.safe_load(f)
    print(f"✓ Loaded config from {config_path}")
    
    # Load data module to get dimensions
    run_output_dir = os.path.join(cfg["output_dir"], cfg["name"])
    data_module_path = os.path.join(run_output_dir, "data_module.torch")
    
    from cell_load.data_modules import PerturbationDataModule
    data_module = PerturbationDataModule.load_state(data_module_path)
    print(f"✓ Loaded data module from {data_module_path}")
    
    # Get checkpoint
    checkpoint_dir = os.path.join(run_output_dir, "checkpoints")
    checkpoint_path = os.path.join(checkpoint_dir, checkpoint_name)
    
    # Load model
    model_class_name = cfg["model"]["name"]
    model_kwargs = cfg["model"]["kwargs"]
    
    print(f"Loading model: {model_class_name}")
    
    if model_class_name.lower() in ["neuralot", "pertsets"]:
        from state.tx.models.pert_sets import PertSetsPerturbationModel
        ModelClass = PertSetsPerturbationModel
    elif model_class_name.lower() == "embedsum":
        from state.tx.models.embed_sum import EmbedSumPerturbationModel
        ModelClass = EmbedSumPerturbationModel
    elif model_class_name.lower() == "decoder_only":
        from state.tx.models.decoder_only import DecoderOnlyPerturbationModel
        ModelClass = DecoderOnlyPerturbationModel
    else:
        raise ValueError(f"Unknown model: {model_class_name}")
    
    var_dims = data_module.get_var_dims()
    
    model_init_kwargs = {
        "input_dim": var_dims["input_dim"],
        "hidden_dim": model_kwargs["hidden_dim"],
        "gene_dim": var_dims["gene_dim"],
        "hvg_dim": var_dims["hvg_dim"],
        "output_dim": var_dims["output_dim"],
        "pert_dim": var_dims["pert_dim"],
        **model_kwargs,
    }
    
    model = ModelClass.load_from_checkpoint(checkpoint_path, **model_init_kwargs)
    model.eval()
    model = model.to(DEVICE)
    
    print(f"✓ Model loaded from {checkpoint_path}")
    print(f"  HVG dim: {var_dims['hvg_dim']}")
    print(f"  Gene dim: {var_dims['gene_dim']}")
    print(f"  Output dim: {var_dims['output_dim']}")
    
    return model, var_dims

In [ ]:
# Load the State model
print("Loading State model...")
model, var_dims = load_state_model(MODEL_DIR, CHECKPOINT)
print("✓ Model ready for decoding")

## Step 2: Decode Embeddings to Gene Expression

In [ ]:
def decode_embeddings_to_genes(model, embeddings, batch_size=512, device='cuda'):
    """Decode embeddings to HVG gene expression space."""
    n_cells = embeddings.shape[0]
    hvg_dim = model.hvg_dim
    
    print(f"Decoding {n_cells:,} cells to {hvg_dim} HVG genes")
    
    # Preallocate output
    decoded = np.empty((n_cells, hvg_dim), dtype=np.float32)
    
    with torch.no_grad():
        for start_idx in tqdm(range(0, n_cells, batch_size), desc="Decoding"):
            end_idx = min(start_idx + batch_size, n_cells)
            batch = embeddings[start_idx:end_idx]
            
            # Convert to tensor
            batch_tensor = torch.from_numpy(batch).float().to(device)
            
            # Decode to genes
            if hasattr(model, 'decode_to_genes'):
                batch_decoded = model.decode_to_genes(batch_tensor)
            elif hasattr(model, 'gene_decoder'):
                batch_decoded = model.gene_decoder(batch_tensor)
            elif hasattr(model, 'decoder'):
                batch_decoded = model.decoder(batch_tensor)
            else:
                raise ValueError("Model has no decoder method")
            
            decoded[start_idx:end_idx] = batch_decoded.cpu().numpy()
    
    print(f"✓ Decoded to shape {decoded.shape}")
    return decoded

In [ ]:
def decode_predictions_file(input_path, output_path, model, embed_key, var_dims, batch_size=512):
    """Load predictions, decode to genes, and save."""
    print(f"\n{'='*60}")
    print(f"Processing: {input_path}")
    print(f"{'='*60}")
    
    # Load predictions
    print("Loading predictions...")
    adata = ad.read_h5ad(input_path)
    print(f"  Shape: {adata.shape}")
    print(f"  obsm keys: {list(adata.obsm.keys())}")
    
    # Get embeddings
    if embed_key not in adata.obsm:
        raise ValueError(f"Embedding key '{embed_key}' not found. Available: {list(adata.obsm.keys())}")
    
    embeddings = adata.obsm[embed_key]
    print(f"  Embedding shape: {embeddings.shape}")
    
    # Decode to gene expression
    decoded_expr = decode_embeddings_to_genes(model, embeddings, batch_size=batch_size, device=DEVICE)
    
    # Create output AnnData with decoded expression in .X
    import pandas as pd
    
    gene_names = var_dims["gene_names"]
    if len(gene_names) != decoded_expr.shape[1]:
        print(f"Warning: gene_names length mismatch, using generic names")
        gene_names = [f"gene_{i}" for i in range(decoded_expr.shape[1])]
    
    var = pd.DataFrame({"gene_names": gene_names})
    
    adata_decoded = ad.AnnData(
        X=decoded_expr,
        obs=adata.obs.copy(),
        var=var
    )
    
    # Keep original embeddings for reference
    adata_decoded.obsm[embed_key] = embeddings
    
    # Save
    print(f"Saving to {output_path}...")
    adata_decoded.write_h5ad(output_path)
    print(f"✓ Saved decoded predictions")
    print(f"  .X shape: {adata_decoded.X.shape} (gene expression)")
    print(f"  .obsm['{embed_key}'] shape: {adata_decoded.obsm[embed_key].shape} (embeddings)")
    
    return output_path

### Decode State Model Predictions

In [ ]:
pred_state_decoded = decode_predictions_file(
    input_path=PRED_STATE,
    output_path=OUTPUT_DIR / "pred_state_decoded.h5ad",
    model=model,
    embed_key=EMBED_KEY_STATE,
    var_dims=var_dims,
    batch_size=BATCH_SIZE
)

### Decode Mean Shift Predictions

In [ ]:
pred_mean_shift_decoded = decode_predictions_file(
    input_path=PRED_MEAN_SHIFT,
    output_path=OUTPUT_DIR / "pred_mean_shift_decoded.h5ad",
    model=model,
    embed_key=EMBED_KEY_BASELINES,
    var_dims=var_dims,
    batch_size=BATCH_SIZE
)

### Decode Control Passthrough Predictions

In [ ]:
pred_control_decoded = decode_predictions_file(
    input_path=PRED_CONTROL,
    output_path=OUTPUT_DIR / "pred_control_decoded.h5ad",
    model=model,
    embed_key=EMBED_KEY_BASELINES,
    var_dims=var_dims,
    batch_size=BATCH_SIZE
)

## Step 3: Run Pearson Correlation Evaluation

Now we'll use the `pearson_delta_only.py` script to evaluate each approach.

In [ ]:
import subprocess

def run_pearson_evaluation(pred_file, output_suffix, real_file=REAL_H5AD):
    """Run Pearson correlation evaluation."""
    outdir = OUTPUT_DIR / f"pearson_{output_suffix}"
    
    cmd = [
        "python", "../scripts/pearson_delta_only.py",
        "--adata-pred", str(pred_file),
        "--adata-real", str(real_file),
        "--control-pert", CONTROL_PERT,
        "--pert-col", PERT_COL,
        "--celltype-col", CELLTYPE_COL,
        "--outdir", str(outdir)
    ]
    
    print(f"\n{'='*60}")
    print(f"Running Pearson evaluation: {output_suffix}")
    print(f"{'='*60}")
    print("Command:", " ".join(cmd))
    
    result = subprocess.run(cmd, capture_output=True, text=True)
    
    if result.returncode != 0:
        print("ERROR:")
        print(result.stderr)
        raise RuntimeError(f"Pearson evaluation failed for {output_suffix}")
    
    print(result.stdout)
    print(f"✓ Results saved to {outdir}")
    
    return outdir

### Evaluate State Model

In [ ]:
pearson_state = run_pearson_evaluation(
    pred_file=pred_state_decoded,
    output_suffix="state_model"
)

### Evaluate Mean Shift

In [ ]:
pearson_mean_shift = run_pearson_evaluation(
    pred_file=pred_mean_shift_decoded,
    output_suffix="mean_shift"
)

### Evaluate Control Passthrough

In [ ]:
pearson_control = run_pearson_evaluation(
    pred_file=pred_control_decoded,
    output_suffix="control_passthrough"
)

## Step 4: Compare Results

In [ ]:
import polars as pl

def load_pearson_results(outdir):
    """Load aggregated Pearson results."""
    agg_results_path = outdir / "agg_results.csv"
    df = pl.read_csv(agg_results_path)
    mean_row = df.filter(pl.col("statistic") == "mean")
    return float(mean_row.select("pearson_delta").item())

# Load results
pearson_state_score = load_pearson_results(pearson_state)
pearson_mean_shift_score = load_pearson_results(pearson_mean_shift)
pearson_control_score = load_pearson_results(pearson_control)

print("\n" + "="*60)
print("FINAL PEARSON CORRELATION RESULTS")
print("="*60)
print(f"State Model:         {pearson_state_score:.4f}")
print(f"Mean Shift:          {pearson_mean_shift_score:.4f}")
print(f"Control Passthrough: {pearson_control_score:.4f}")
print("="*60)

# Calculate differences
print("\nDifferences:")
print(f"State - Mean Shift:  {pearson_state_score - pearson_mean_shift_score:+.4f}")
print(f"State - Control:     {pearson_state_score - pearson_control_score:+.4f}")
print(f"Mean Shift - Control: {pearson_mean_shift_score - pearson_control_score:+.4f}")

## Summary

This notebook:
1. ✓ Loaded the State model with its gene decoder
2. ✓ Decoded all 3 prediction files from embedding space to gene expression space
3. ✓ Ran Pearson correlation evaluation comparing predicted vs real gene expression
4. ✓ Compared results across State model, mean shift, and control passthrough

The Pearson correlation metric measures how well each approach predicts gene expression changes caused by perturbations (cytokines).